In [27]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/base.xlsx")

In [4]:
for col in df.columns:
    pct_missing = np.mean(df[col].isnull())
    print('{} - {}%'.format(col, round(pct_missing*100)))

add_certificates - 92%
additional_skills - 37%
birthday - 77%
birthday_mistake - 75%
business_trips - 1%
busy_type - 0%
country - 0%
date_creation - 0%
date_inactivation - 8%
date_last_updated - 0%
date_modify_inner_info - 0%
date_publish - 0%
date_time_publish - 0%
drive_licences - 64%
driver_licence_a - 0%
driver_licence_b - 0%
driver_licence_c - 0%
driver_licence_d - 0%
driver_licence_e - 0%
education_type - 81%
experience - 0%
experience_mistake - 0%
gender - 30%
id_candidate - 0%
id_cv - 0%
id_user_inner_info - 0%
inactive - 0%
industry_code - 0%
inner_info_fullness_rate - 0%
inner_info_status - 0%
locality - 0%
nark_certificate - 100%
other_info - 86%
position_name - 0%
profession_code - 53%
region_code - 0%
relocation - 5%
retraining_capability - 8%
salary - 0%
schedule_type - 0%
schedule_type_1 - 0%
schedule_type_2 - 0%
schedule_type_3 - 0%
schedule_type_4 - 0%
schedule_type_5 - 0%
schedule_type_6 - 0%
skills - 50%
time_publish - 0%
worldskills_international_name - 100%
worldsk

In [39]:
import pandas as pd


def cast_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Очистка текстовых псевдо-пропусков
    df = df.replace(
        [r"^\s*$", "NaN", "nan", "None", "null", "NULL", "<NA>", "N/A", "n/a"],
        pd.NA,
        regex=True,
    )

    # 2. Возраст из года рождения
    if "birthday" in df.columns:
        s = pd.to_numeric(df["birthday"], errors="coerce")
        df["birthday"] = (2026 - s).astype("Int64")
        df = df.rename(columns={"birthday": "age"})

    # 3. region_code из КЛАДР-кода locality
    # Предполагаем, что locality содержит код КЛАДР вида 2300000000000
    if "locality" in df.columns:
        kladr = (
            df["locality"]
            .astype("string")
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

        df["region_code"] = pd.to_numeric(
            kladr.str[:2],
            errors="coerce"
        ).astype("Int64")

    string_cols = [
        "gender",
        "country",
        "skills",
        "additional_skills",
        "education_type",
        "other_info",
        "industry_code",
        "busy_type",
        "position_name",
        "schedule_type",
        "region_code",  # если хочешь хранить как строку, убери из int-логики ниже
    ]

    int_cols = [
        "experience",
        "salary",
        "inner_info_fullness_rate",
        "region_code",  # если нужен именно числовой код региона
    ]

    bool_like_int_cols = [
        "business_trips",
        "relocation",
        "retraining_capability",
    ]

    # Строковые колонки
    for col in [
        "gender",
        "country",
        "skills",
        "additional_skills",
        "education_type",
        "other_info",
        "industry_code",
        "busy_type",
        "position_name",
        "schedule_type",
    ]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Целочисленные колонки
    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # Булевы флаги как 0/1/NA
    for col in bool_like_int_cols:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce")
            s = s.where(s.isin([0, 1]) | s.isna())
            df[col] = s.astype("Int64")

    return df


def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    mask = pd.Series(True, index=df.index)

    if "business_trips" in df.columns:
        mask &= df["business_trips"] == 0

    if "busy_type" in df.columns:
        mask &= df["busy_type"] == "Полная занятость"

    if "country" in df.columns:
        mask &= df["country"] == "Российская Федерация"

    if "industry_code" in df.columns:
        mask &= df["industry_code"] == "BuldindRealty"   # проверь написание

    if "relocation" in df.columns:
        mask &= df["relocation"] == 0

    if "retraining_capability" in df.columns:
        mask &= df["retraining_capability"] == 1

    if "schedule_type" in df.columns:
        mask &= df["schedule_type"] == "Полный рабочий день"

    return df.loc[mask].copy()


def drop_unused_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    cols_to_drop = [
        "add_certificates",
        "nark_certificate",
        "worldskills_international_name",
        "worldskills_is_international",
        "worldskills_russian_name",
        "worldskills_skill_abbreviation",
        "worldskills_type",
        "locality",
        "id_candidate",
        "id_cv",
        "id_user_inner_info",
        "experience_mistake",
        "birthday_mistake",
        "date_creation",
        "date_inactivation",
        "date_last_updated",
        "date_modify_inner_info",
        "date_publish",
        "date_time_publish",
        "drive_licences",
        "driver_licence_a",
        "driver_licence_b",
        "driver_licence_c",
        "driver_licence_d",
        "driver_licence_e",
        "inner_info_status",
        "profession_code",
        "time_publish",
        "worldskills_inspection_status",
        "inactive",
        "schedule_type_1",
        "schedule_type_2",
        "schedule_type_3",
        "schedule_type_4",
        "schedule_type_5",
        "schedule_type_6",
    ]

    return df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")


df = cast_columns(df)
df = apply_filters(df)
df = drop_unused_columns(df)

print(df)
df.to_excel("../data/clean_base.xlsx", index=False)

                                       additional_skills   age  \
70                                                  <NA>  <NA>   
86     Добросовестность, усидчивость, желание работат...  <NA>   
103                                                 <NA>  <NA>   
128                                                 <NA>  <NA>   
172    Ответственность, внимательность, стресоустойчи...  <NA>   
...                                                  ...   ...   
49883                                               <NA>  <NA>   
49923             Ответственная, усидчивая, пунктуальная  <NA>   
49925  Внимательность, ответственность, целеустремлен...  <NA>   
49946  В работе ответственен, исполнителен, умения ра...  <NA>   
49978  Энергичный,целеустремленный,быстро обучаемый и...  <NA>   

       business_trips         busy_type               country education_type  \
70                  0  Полная занятость  Российская Федерация           <NA>   
86                  0  Полная занятость  Россий

In [23]:
import pandas as pd

def check_column_types(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in df.columns:
        non_null = df[col].dropna()

        python_types = sorted(
            {type(x).__name__ for x in non_null},
        )

        rows.append({
            "column": col,
            "pandas_dtype": str(df[col].dtype),
            "non_null_count": int(non_null.shape[0]),
            "null_count": int(df[col].isna().sum()),
            "python_types_in_values": ", ".join(python_types) if python_types else "no non-null values"
        })

    return pd.DataFrame(rows)


report = check_column_types(df)
print(report)

                           column    pandas_dtype  non_null_count  null_count  \
0               additional_skills          string           31388       18611   
1                        birthday           Int64           11627       38372   
2                birthday_mistake           Int64               0       49999   
3                  business_trips           Int64               0       49999   
4                       busy_type          string           49999           0   
5                         country          string           49999           0   
6                   date_creation  datetime64[us]           49986          13   
7               date_inactivation  datetime64[us]           45900        4099   
8               date_last_updated  datetime64[us]           49999           0   
9          date_modify_inner_info  datetime64[us]           49986          13   
10                   date_publish  datetime64[us]           49999           0   
11              date_time_pu